# IoT Network Intrusion Detection - Data Exploration

This notebook explores the BoT-IoT dataset to understand its characteristics, feature distributions, and class balance.

## Dataset Information
- **Dataset**: BoT-IoT (UNSW Canberra Cyber Range)
- **Files**: 4 CSV files with 10 best features
- **Total Records**: ~3 million
- **Features**: 10 network features
- **Classes**: 5 attack types (Normal, DDoS, DoS, Reconnaissance, Theft)
 

In [ ]:
# Import necessary libraries
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

# Add src directory to path
sys.path.append(str(Path().resolve().parent / "src"))

from data_loader import IoTDataLoader

# Set style and ignore warnings
plt.style.use('default')
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

print("Libraries imported successfully!")


## 1. Load Dataset


In [ ]:
# Load the dataset
data_path = "../data/raw"
loader = IoTDataLoader(data_path)

print("Loading BoT-IoT dataset...")
data = loader.load_data()
print("Dataset loaded successfully!")

# Get dataset information
info = loader.get_data_info()
print(f"\nDataset Information:")
print(f"Total Records: {info['total_records']:,}")
print(f"Total Features: {info['total_features']}")
print(f"Memory Usage: {info['memory_usage']:.2f} MB")
print(f"Number of Classes: {info.get('num_classes', 'Unknown')}")


## 2. Basic Dataset Information


In [ ]:
# Display basic information about the dataset
print("Dataset Shape:", data.shape)
print("\nColumn Names:")
for i, col in enumerate(data.columns, 1):
    print(f"{i:2d}. {col}")

print("\nData Types:")
print(data.dtypes)

print("\nMissing Values:")
missing_values = data.isnull().sum()
print(missing_values[missing_values > 0] if missing_values.sum() > 0 else "No missing values found")


In [ ]:
# Display first few rows
print("First 5 rows of the dataset:")
display(data.head())

print("\nLast 5 rows of the dataset:")
display(data.tail())


## 3. Target Variable Analysis


In [ ]:
# Analyze the target variable (category)
print("Target Variable Analysis:")
print("=" * 40)

# Class distribution
class_counts = data['category'].value_counts()
class_percentages = data['category'].value_counts(normalize=True) * 100

print("\nClass Distribution:")
for class_name, count in class_counts.items():
    percentage = class_percentages[class_name]
    print(f"{class_name:15s}: {count:8,} ({percentage:6.2f}%)")

print(f"\nTotal Classes: {len(class_counts)}")
print(f"Total Samples: {len(data):,}")


In [ ]:
# Create class distribution visualization
plt.figure(figsize=(15, 6))

# Bar chart
plt.subplot(1, 2, 1)
bars = plt.bar(class_counts.index, class_counts.values, 
               color=['skyblue', 'lightcoral', 'lightgreen', 'orange', 'purple'])
plt.title('Attack Type Distribution (Count)', fontsize=14, fontweight='bold')
plt.xlabel('Attack Types', fontsize=12)
plt.ylabel('Number of Samples', fontsize=12)
plt.xticks(rotation=45, ha='right')

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height):,}', ha='center', va='bottom', fontsize=10)

# Pie chart
plt.subplot(1, 2, 2)
colors = ['skyblue', 'lightcoral', 'lightgreen', 'orange', 'purple']
wedges, texts, autotexts = plt.pie(class_counts.values, labels=class_counts.index, 
                                   autopct='%1.1f%%', colors=colors, startangle=90)
plt.title('Attack Type Distribution (Percentage)', fontsize=14, fontweight='bold')

# Improve text readability
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')

plt.tight_layout()
plt.show()


## 4. Feature Analysis


In [ ]:
# Get feature columns (exclude target variable)
feature_columns = [col for col in data.columns if col != 'category']
print(f"Feature columns ({len(feature_columns)}): {feature_columns}")

# Statistical summary of features
print("\nStatistical Summary of Features:")
print("=" * 50)
display(data[feature_columns].describe())


In [ ]:
# Identify numeric features for visualization
numeric_features = data.select_dtypes(include=['number']).columns.tolist()
# Remove non-feature columns
exclude_columns = ['Unnamed: 0', 'pkSeqID', 'attack']
numeric_features = [col for col in numeric_features if col not in exclude_columns]

print(f"Numeric Features ({len(numeric_features)}):")
for i, feature in enumerate(numeric_features, 1):
    print(f"{i:2d}. {feature}")

# Create feature distribution plots for numeric features
n_features = len(numeric_features)
n_cols = 3
n_rows = (n_features + n_cols - 1) // n_cols

plt.figure(figsize=(15, 5 * n_rows))

for i, feature in enumerate(numeric_features, 1):
    plt.subplot(n_rows, n_cols, i)
    
    # Create histogram
    plt.hist(data[feature], bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    plt.title(f'{feature} Distribution', fontsize=12, fontweight='bold')
    plt.xlabel(feature, fontsize=10)
    plt.ylabel('Frequency', fontsize=10)
    plt.grid(True, alpha=0.3)
    
    # Add statistics
    mean_val = data[feature].mean()
    std_val = data[feature].std()
    plt.axvline(mean_val, color='red', linestyle='--', alpha=0.8, label=f'Mean: {mean_val:.2f}')
    plt.axvline(mean_val + std_val, color='orange', linestyle='--', alpha=0.6, label=f'±1σ: {std_val:.2f}')
    plt.axvline(mean_val - std_val, color='orange', linestyle='--', alpha=0.6)
    plt.legend(fontsize=8)

plt.tight_layout()
plt.show()


## 5. Correlation Analysis


In [ ]:
# Calculate correlation matrix for numeric features
correlation_matrix = data[numeric_features].corr()

# Create correlation heatmap
plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))  # Mask upper triangle

sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": .8}, fmt='.3f')

plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Find highly correlated feature pairs
print("\nHighly Correlated Feature Pairs (|r| > 0.7):")
print("=" * 50)

high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr_val = correlation_matrix.iloc[i, j]
        if abs(corr_val) > 0.7:
            high_corr_pairs.append((
                correlation_matrix.columns[i], 
                correlation_matrix.columns[j], 
                corr_val
            ))

if high_corr_pairs:
    for feat1, feat2, corr in high_corr_pairs:
        print(f"{feat1:20s} <-> {feat2:20s}: {corr:6.3f}")
else:
    print("No highly correlated feature pairs found.")


## 6. Attack Pattern Analysis


In [ ]:
# Analyze feature patterns for each attack type
print("Feature Statistics by Attack Type:")
print("=" * 50)

# Group by attack type and calculate statistics
attack_stats = data.groupby('category')[numeric_features].agg(['mean', 'std', 'min', 'max']).round(4)
display(attack_stats)


In [ ]:
# Create box plots for feature distributions by class (using first 6 features)
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

# Select 6 most important features for visualization
important_features = numeric_features[:6]

for i, feature in enumerate(important_features):
    ax = axes[i]
    
    # Create box plot
    data.boxplot(column=feature, by='category', ax=ax)
    ax.set_title(f'{feature} by Attack Type', fontsize=12, fontweight='bold')
    ax.set_xlabel('Attack Type', fontsize=10)
    ax.set_ylabel(feature, fontsize=10)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3)

# Remove the automatic title from boxplot
plt.suptitle('Feature Distributions by Attack Type', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


## 7. Data Quality Assessment


In [ ]:
# Data quality assessment
print("Data Quality Assessment:")
print("=" * 40)

# Check for duplicates
duplicates = data.duplicated().sum()
print(f"Duplicate rows: {duplicates:,} ({duplicates/len(data)*100:.2f}%)")

# Check for outliers using IQR method
print("\nOutlier Detection (IQR Method):")
outlier_summary = {}

for feature in numeric_features:
    Q1 = data[feature].quantile(0.25)
    Q3 = data[feature].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = data[(data[feature] < lower_bound) | (data[feature] > upper_bound)][feature]
    outlier_count = len(outliers)
    outlier_percentage = (outlier_count / len(data)) * 100
    
    outlier_summary[feature] = {
        'count': outlier_count,
        'percentage': outlier_percentage,
        'lower_bound': lower_bound,
        'upper_bound': upper_bound
    }
    
    if outlier_count > 0:
        print(f"{feature:20s}: {outlier_count:8,} outliers ({outlier_percentage:5.1f}%)")

# Check data consistency
print("\nData Consistency Checks:")
print(f"Negative values in numeric features:")
for feature in numeric_features:
    negative_count = (data[feature] < 0).sum()
    if negative_count > 0:
        print(f"  {feature}: {negative_count:,} negative values")

# Memory usage analysis
print(f"\nMemory Usage Analysis:")
print(f"Total memory usage: {data.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Memory per record: {data.memory_usage(deep=True).sum() / len(data) / 1024:.2f} KB")


## 8. Summary and Key Insights


In [ ]:
# Generate summary insights
print("KEY INSIGHTS FROM DATA EXPLORATION:")
print("=" * 50)

print(f"\n1. Dataset Overview:")
print(f"   • Total records: {len(data):,}")
print(f"   • Total features: {len(feature_columns)}")
print(f"   • Memory usage: {data.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"   • Data quality: {'Good' if duplicates == 0 else 'Issues found'}")

print(f"\n2. Class Distribution:")
for class_name, count in class_counts.items():
    percentage = class_percentages[class_name]
    print(f"   • {class_name:15s}: {percentage:5.1f}% ({count:,} samples)")

print(f"\n3. Feature Characteristics:")
print(f"   • Numeric features: {len(numeric_features)}")
print(f"   • Highly correlated pairs: {len(high_corr_pairs)}")
print(f"   • Duplicate records: {duplicates:,}")

print(f"\n4. Data Quality:")
print(f"   • Missing values: {'None found' if missing_values.sum() == 0 else 'Present'}")
print(f"   • Outliers detected: {sum(1 for v in outlier_summary.values() if v['count'] > 0)} features")

print(f"\n5. Recommendations for Preprocessing:")
print(f"   • Feature scaling required: Yes (due to different scales)")
print(f"   • Class imbalance handling: {'Yes' if max(class_percentages) - min(class_percentages) > 10 else 'No'}")
print(f"   • Outlier treatment: {'Consider capping' if sum(1 for v in outlier_summary.values() if v['count'] > 0) > 0 else 'Not critical'}")

print(f"\n6. Model Training Considerations:")
print(f"   • Dataset size: {'Large' if len(data) > 1000000 else 'Medium'}")
print(f"   • Training time estimate: {'High' if len(data) > 1000000 else 'Medium'}")
print(f"   • Memory requirements: {'High' if data.memory_usage(deep=True).sum() / 1024**2 > 100 else 'Medium'}")

print(f"\nData exploration completed successfully!")
print(f"Ready for preprocessing and model training.")
